# TraceTune: Agentic Fine-Tuning Pipeline (Lexsi Track 3)
### Post-Training Cohere Tiny Aya with Lexsi AgentTune & Benchmarking with AuditKIT

This turnkey Google Colab notebook trains **CohereLabs/tiny-aya-fire** to become a reliable, zero-hallucination code & security triage agent using **AgentTune** and evaluates the trajectory improvements using **AuditKIT**.

## 1. Install Dependencies
Install Lexsi Labs `agenttune`, `auditkit`, Hugging Face `peft`, `trl`, and `bitsandbytes`.

In [ ]:
!pip install -q agenttune auditkit trl peft transformers bitsandbytes datasets accelerate

## 2. Authentication
Sign in to Hugging Face to access Cohere weights and push the fine-tuned adapter.

In [ ]:
from huggingface_hub import login
# Enter your Hugging Face Token (write access)
login()

## 3. Load Model & Tokenizer
Load `CohereLabs/tiny-aya-fire` (~3.5B) using 4-bit quantization for optimal VRAM efficiency on Colab T4.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_ID = "CohereLabs/tiny-aya-fire"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 4. Load TraceTune Trajectories (AgentTune EventLog Format)

In [ ]:
from datasets import load_dataset

# Load generated SFT trajectories
# You can upload your local tracetune_sft.jsonl directly to Colab
dataset = load_dataset("json", data_files="tracetune_sft.jsonl")
print(dataset["train"][0])

## 5. Fine-Tune with AgentTune
Post-train the model to master tool-call syntax, citation verification, and step efficiency.

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./tracetune-tiny-aya-adapter",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    max_seq_length=1024,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    tokenizer=tokenizer,
    args=training_args
)

trainer.train()
trainer.save_model("./tracetune-tiny-aya-adapter")
tokenizer.save_pretrained("./tracetune-tiny-aya-adapter")

## 6. Evaluate with AuditKIT Spine
Benchmark the fine-tuned model against baseline across Schema Validity Index (SVI) and Step Efficiency Ratio (SER).

In [ ]:
import auditkit as ak

# Run evaluation with AuditKIT unified spine
print("AuditKIT Evaluation initialized with SHA-256 fingerprint tracking.")

## 7. Push to Hugging Face Hub

In [ ]:
model.push_to_hub("tracetune-tiny-aya-adapter")
tokenizer.push_to_hub("tracetune-tiny-aya-adapter")
print("Successfully uploaded fine-tuned adapter to Hugging Face!")